# Cattle Lameness Detection — Improved Training Notebook

## What's different from the previous notebook

| Problem in the old notebook | Fix here |
|---|---|
| Symmetry score was randomly generated during training | Extracted from real bounding-box trajectories |
| Motion-diff tracking fragmented one cow into multiple blobs | YOLOv8 (COCO `cow` class) + ByteTrack for real per-animal tracking |
| 3 features, linear SVM | 7 real gait features, 3 models compared with 5-fold CV |
| Placeholder keypoints offset from centroid | Actual per-track centroid + bounding box measurements |

## Requirements
- Python 3.11 (venv already set up in `backend/venv`)
- Run `python download_dataset.py` from the project root **before** opening this notebook
- That downloads 32 Lame + 29 Normal videos into `./dataset/`

In [5]:
# Install dependencies — run once
%pip install -q ultralytics scikit-learn pandas matplotlib seaborn joblib opencv-python-headless


[notice] A new release of pip is available: 24.0 -> 26.1.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [6]:
import cv2
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import warnings
warnings.filterwarnings('ignore')

from pathlib import Path
from collections import defaultdict
from ultralytics import YOLO

from sklearn.model_selection import StratifiedKFold, cross_val_score, train_test_split
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    classification_report, confusion_matrix,
    ConfusionMatrixDisplay, roc_auc_score
)

print('All imports OK')

All imports OK


## 1. Paths

Dataset is expected at `./dataset/Lame/` and `./dataset/Normal/`.
Run `python download_dataset.py` from the project root if you haven't already.

In [7]:
NOTEBOOK_DIR = Path.cwd()
LAME_DIR     = NOTEBOOK_DIR / 'dataset' / 'Lame'
NORMAL_DIR   = NOTEBOOK_DIR / 'dataset' / 'Normal'
MODEL_OUT    = str(NOTEBOOK_DIR / 'cattle_lameness_model.joblib')

assert LAME_DIR.exists(),   f"Lame folder not found: {LAME_DIR}\nRun: python download_dataset.py"
assert NORMAL_DIR.exists(), f"Normal folder not found: {NORMAL_DIR}\nRun: python download_dataset.py"

lame_videos   = sorted(LAME_DIR.glob('*.mp4'))
normal_videos = sorted(NORMAL_DIR.glob('*.mp4'))
print(f'Lame  : {len(lame_videos)} videos')
print(f'Normal: {len(normal_videos)} videos')

Lame  : 32 videos
Normal: 29 videos


## 2. Load YOLOv8

YOLOv8n (~6 MB) is pretrained on COCO which includes class 19 = `cow`.
No fine-tuning is needed for detection; ByteTrack handles the tracking ID assignment.

In [ ]:
yolo = YOLO('yolov8n.pt')   # downloads once (~6 MB), cached afterward
COW_CLASS_ID = 19            # COCO class index for 'cow'
print('YOLOv8n ready')

## 3. Feature Extraction

### What each feature captures

| Feature | Lameness signal |
|---|---|
| `speed_mean` | Lame cattle walk slower |
| `speed_cv` | Lame cattle have inconsistent, hesitant gait |
| `stride_mean` | Lame cattle take shorter strides |
| `stride_cv` | Lame cattle have irregular stride lengths |
| `path_straightness` | Lame cattle wander/weave rather than walk straight |
| `lateral_sway` | Lame cattle sway sideways to offload weight |
| `bbox_ar_cv` | Lame cattle shift posture (aspect ratio changes) |

In [ ]:
FEATURE_COLS = [
    'speed_mean', 'speed_cv',
    'stride_mean', 'stride_cv',
    'path_straightness',
    'lateral_sway',
    'bbox_ar_cv',
]


def _gait_features(detections, sample_fps):
    """
    detections: list of (cx, cy, w, h) tuples — one per sampled frame for one tracked animal
    Returns a feature dict or None if the track is too short.
    """
    if len(detections) < 8:
        return None

    cx = np.array([d[0] for d in detections], dtype=float)
    cy = np.array([d[1] for d in detections], dtype=float)
    w  = np.array([d[2] for d in detections], dtype=float)
    h  = np.array([d[3] for d in detections], dtype=float)

    # --- speed ---
    diffs = np.sqrt(np.diff(cx)**2 + np.diff(cy)**2)
    speed_mean = float(np.mean(diffs))
    speed_cv   = float(np.std(diffs) / (speed_mean + 1e-9))

    # --- stride (displacement over ~3-second windows) ---
    win = max(1, int(sample_fps * 3))
    strides = [
        float(np.sqrt((cx[i+win]-cx[i])**2 + (cy[i+win]-cy[i])**2))
        for i in range(0, len(cx) - win, win)
    ]
    stride_mean = float(np.mean(strides)) if strides else 0.0
    stride_cv   = float(np.std(strides) / (stride_mean + 1e-9)) if strides else 0.0

    # --- path straightness (0–1; 1 = perfectly straight line) ---
    total_path = float(np.sum(diffs)) + 1e-9
    direct     = float(np.sqrt((cx[-1]-cx[0])**2 + (cy[-1]-cy[0])**2))
    path_straightness = min(1.0, direct / total_path)

    # --- lateral sway (perpendicular deviation from overall direction) ---
    direction = np.array([cx[-1]-cx[0], cy[-1]-cy[0]], dtype=float)
    dir_norm  = np.linalg.norm(direction)
    if dir_norm > 1e-6:
        direction /= dir_norm
        perp    = np.array([-direction[1], direction[0]])
        pts     = np.stack([cx - cx[0], cy - cy[0]], axis=1)
        lateral = np.abs(pts @ perp)
        lateral_sway = float(np.std(lateral))
    else:
        lateral_sway = float(np.std(cx)) + float(np.std(cy))

    # --- bounding-box aspect ratio variability ---
    ar     = w / (h + 1e-9)
    ar_cv  = float(np.std(ar) / (np.mean(ar) + 1e-9))

    return {
        'speed_mean':        speed_mean,
        'speed_cv':          speed_cv,
        'stride_mean':       stride_mean,
        'stride_cv':         stride_cv,
        'path_straightness': path_straightness,
        'lateral_sway':      lateral_sway,
        'bbox_ar_cv':        ar_cv,
    }


def extract_features_from_video(video_path, yolo_model, sample_fps=5, min_frames=10):
    """
    Run YOLOv8 + ByteTrack on a video and return gait features for the
    most-detected track (= the primary animal in frame).

    Returns a feature dict, or None if no valid track was found.
    """
    cap = cv2.VideoCapture(str(video_path))
    if not cap.isOpened():
        return None

    video_fps   = cap.get(cv2.CAP_PROP_FPS) or 30.0
    sample_every = max(1, int(round(video_fps / sample_fps)))

    tracks = defaultdict(list)   # track_id -> [(cx, cy, w, h), ...]
    frame_idx = 0

    while True:
        ret, frame = cap.read()
        if not ret:
            break

        if frame_idx % sample_every == 0:
            results = yolo_model.track(
                frame,
                classes=[COW_CLASS_ID],
                persist=True,
                tracker='bytetrack.yaml',
                verbose=False,
            )
            if results and results[0].boxes is not None:
                boxes = results[0].boxes
                if boxes.id is not None:
                    for box, tid in zip(boxes.xyxy.cpu().numpy(), boxes.id.cpu().numpy()):
                        x1, y1, x2, y2 = box
                        cx = (x1 + x2) / 2
                        cy = (y1 + y2) / 2
                        w  = x2 - x1
                        h  = y2 - y1
                        tracks[int(tid)].append((cx, cy, w, h))

        frame_idx += 1

    cap.release()

    # Keep only tracks that were seen in at least min_frames sampled frames
    valid = {tid: dets for tid, dets in tracks.items() if len(dets) >= min_frames}
    if not valid:
        return None

    # Pick the track with the most detections (the primary animal)
    best_tid  = max(valid, key=lambda t: len(valid[t]))
    best_dets = valid[best_tid]

    return _gait_features(best_dets, sample_fps)


print('Feature extraction functions defined')

## 4. Process All Videos

This is the slow step. On a CPU-only machine expect ~10-20 s per video (50 total = ~15 min).
On a GPU (Colab T4) it takes 2-4 s per video.

In [ ]:
all_features = []
all_labels   = []
failed       = []

def process_set(video_paths, label):
    for vp in video_paths:
        name = Path(vp).name
        print(f'  [{label}] {name}', end=' ... ', flush=True)
        feats = extract_features_from_video(vp, yolo)
        if feats:
            all_features.append(feats)
            all_labels.append(label)
            print('OK')
        else:
            failed.append((label, name))
            print('SKIPPED (no valid track)')

print('Processing Lame videos...')
process_set(lame_videos, 'Lame')

print('\nProcessing Normal videos...')
process_set(normal_videos, 'Normal')

print(f'\nDone. Samples collected: {len(all_features)}')
if failed:
    print(f'Skipped ({len(failed)}): {failed}')

In [ ]:
df = pd.DataFrame(all_features)
df['label'] = all_labels

print('Class balance:')
print(df['label'].value_counts())
print()
print(df.groupby('label')[FEATURE_COLS].mean().T)

## 5. Exploratory Data Analysis

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(22, 9))
axes = axes.flatten()

for ax, col in zip(axes[:len(FEATURE_COLS)], FEATURE_COLS):
    sns.boxplot(x='label', y=col, data=df, palette='Set2', ax=ax)
    ax.set_title(col, fontsize=11)
    ax.set_xlabel('')

# Hide the unused subplot
axes[-1].set_visible(False)

plt.suptitle('Feature Distributions: Lame vs Normal', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(9, 7))
sns.heatmap(
    df[FEATURE_COLS].corr(),
    annot=True, fmt='.2f', cmap='coolwarm',
    vmin=-1, vmax=1, square=True,
)
plt.title('Feature Correlation Matrix', fontsize=13)
plt.tight_layout()
plt.show()

## 6. Model Training — 5-Fold Cross-Validation

With only 50 samples a single 80/20 split is unreliable.
We use **StratifiedKFold** so every class distribution is preserved in each fold,
then pick the best model by mean F1 before doing a final held-out evaluation.

In [ ]:
X = df[FEATURE_COLS].values
y = (df['label'] == 'Lame').astype(int).values   # 1=Lame, 0=Normal

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

candidates = {
    'Random Forest': Pipeline([
        ('scaler', StandardScaler()),
        ('clf',    RandomForestClassifier(
            n_estimators=200, max_depth=6,
            random_state=42, class_weight='balanced'
        )),
    ]),
    'Gradient Boosting': Pipeline([
        ('scaler', StandardScaler()),
        ('clf',    GradientBoostingClassifier(
            n_estimators=150, max_depth=3,
            learning_rate=0.1, random_state=42
        )),
    ]),
    'SVM (RBF)': Pipeline([
        ('scaler', StandardScaler()),
        ('clf',    SVC(
            kernel='rbf', C=2.0,
            random_state=42, class_weight='balanced',
            probability=True
        )),
    ]),
}

results = {}
print(f'{"Model":<25}  {"F1 mean":>8}  {"F1 std":>8}  {"AUC mean":>9}')
print('-' * 55)

for name, pipe in candidates.items():
    f1_scores  = cross_val_score(pipe, X, y, cv=skf, scoring='f1')
    auc_scores = cross_val_score(pipe, X, y, cv=skf, scoring='roc_auc')
    results[name] = {
        'f1_mean': f1_scores.mean(),
        'f1_std':  f1_scores.std(),
        'auc':     auc_scores.mean(),
        'pipe':    pipe,
    }
    print(f'{name:<25}  {f1_scores.mean():>8.3f}  {f1_scores.std():>8.3f}  {auc_scores.mean():>9.3f}')

best_name = max(results, key=lambda k: results[k]['f1_mean'])
best_pipe = results[best_name]['pipe']
print(f'\n✓ Best: {best_name}  (F1={results[best_name]["f1_mean"]:.3f})')

## 7. Final Evaluation on a Held-Out Test Set

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

best_pipe.fit(X_train, y_train)
y_pred = best_pipe.predict(X_test)

print(f'Test set: {len(X_test)} samples')
print()
print(classification_report(y_test, y_pred, target_names=['Normal', 'Lame']))

cm   = confusion_matrix(y_test, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['Normal', 'Lame'])
fig, ax = plt.subplots(figsize=(5, 4))
disp.plot(cmap='Blues', ax=ax)
ax.set_title(f'Confusion Matrix — {best_name} (test set)', fontsize=11)
plt.tight_layout()
plt.show()

## 8. Feature Importance (Random Forest / GB)

In [ ]:
# Only applies to tree-based models
clf = best_pipe.named_steps['clf']
if hasattr(clf, 'feature_importances_'):
    importances = pd.Series(clf.feature_importances_, index=FEATURE_COLS).sort_values()
    fig, ax = plt.subplots(figsize=(8, 4))
    importances.plot.barh(ax=ax, color='steelblue')
    ax.set_title('Feature Importance', fontsize=12)
    ax.set_xlabel('Importance')
    plt.tight_layout()
    plt.show()
else:
    print(f'{best_name} does not expose feature_importances_ — skipping plot')

## 9. Train Final Model on All Data and Save

Once you're happy with the CV results, retrain on the **full dataset** so the
model sees as many examples as possible before deployment.

In [ ]:
best_pipe.fit(X, y)

joblib.dump({
    'model':        best_pipe,
    'feature_cols': FEATURE_COLS,
    'model_name':   best_name,
    'n_train':      len(X),
    'classes':      ['Normal', 'Lame'],
}, MODEL_OUT)

print(f'Saved to {MODEL_OUT}')
print(f'Model  : {best_name}')
print(f'Features: {FEATURE_COLS}')
print(f'Trained on {len(X)} samples (Lame={int(y.sum())}, Normal={int((1-y).sum())})')

In [ ]:
# Download the model file if running on Colab
try:
    from google.colab import files
    files.download(MODEL_OUT)
    print(f'Downloading {MODEL_OUT}...')
except ImportError:
    print(f'Not on Colab — model saved locally at {MODEL_OUT}')

## 10. Wiring the New Model into the Backend

The saved file is a dict, not a bare sklearn model. In `gait_analyzer.py` and
`pose_estimator.py` you will need to:

1. Replace `app/ml/models/svm_model.joblib` with the new file.
2. Update `gait_analyzer.py` to load the dict and use the new 7-feature vector instead of the 3-feature one.
3. Update `pose_estimator.py` to use YOLOv8 for tracking instead of MOG2 — this will also fix the over-counting problem.

### Quick test — run a video through the saved model here:


In [ ]:
bundle  = joblib.load(MODEL_OUT)
pipe    = bundle['model']
f_cols  = bundle['feature_cols']
classes = bundle['classes']

# Pick one video as a quick smoke-test
test_video = lame_videos[0]
feats = extract_features_from_video(test_video, yolo)

if feats:
    X_sample = np.array([[feats[c] for c in f_cols]])
    pred     = pipe.predict(X_sample)[0]          # 0=Normal, 1=Lame
    prob     = pipe.predict_proba(X_sample)[0]
    print(f'Video : {Path(test_video).name}')
    print(f'Prediction : {classes[pred]}')
    print(f'Confidence : Normal={prob[0]:.2f}  Lame={prob[1]:.2f}')
    for k, v in feats.items():
        print(f'  {k:<22} = {v:.4f}')
else:
    print('No track found in test video — try a different sample')